In [ ]:
#@title 1 · Setup & configuration
import os, sys, json, time, math, random, hashlib, itertools, copy, gc, warnings, zipfile, datetime, traceback
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

# ===================== USER SETTINGS =====================
PRESET      = "full"     # "quick" | "standard" | "full"   (full = 5 seeds, as the protocol asks)
MAX_HOURS   = 9.0        # stop launching new runs after this wall-clock budget (re-run cells to resume)
USE_DRIVE   = True       # store results in Google Drive so they survive disconnects
RESULTS_DIR_NAME = "socr_results"
FACTORIAL   = False      # also run the full 144-config factorial ablation (very long)
EXPERIMENTS = dict(E1=True, E2=True, E3=True, E4=True, E5=True, E6=True, E7=True, E8=True)
# ==========================================================

PRESETS = {
 "quick":    dict(seeds=[0,1],         pm_tasks=8,  pm_tasks_long=12, pm_ntrain=3000, pm_epochs=3, cifar_tasks=[10],    cifar_epochs=6,  widths=[64,128],     n_R=600),
 "standard": dict(seeds=[0,1,2],       pm_tasks=15, pm_tasks_long=24, pm_ntrain=5000, pm_epochs=4, cifar_tasks=[10],    cifar_epochs=10, widths=[64,128,256], n_R=1000),
 "full":     dict(seeds=[0,1,2,3,4],   pm_tasks=20, pm_tasks_long=30, pm_ntrain=6000, pm_epochs=5, cifar_tasks=[10,20], cifar_epochs=10, widths=[64,128,256], n_R=1000),
}
P = PRESETS[PRESET]

OUT = "/content/socr_results"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT = "/content/drive/MyDrive/" + RESULTS_DIR_NAME
    except Exception as e:
        print("Drive not available -> using local disk:", e)
OUT = os.path.join(OUT, PRESET)
os.makedirs(OUT, exist_ok=True); os.makedirs(os.path.join(OUT, "figs"), exist_ok=True)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEV.type == "cpu": print("WARNING: no GPU detected - this will be very slow. Runtime > Change runtime type > GPU.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
F64 = torch.float64
print("device:", DEV, "| preset:", PRESET, "| output dir:", OUT)
print("torch", torch.__version__)


In [ ]:
#@title 2 · Data & benchmarks
import torchvision
DATA_ROOT = "/content/data"
_BASE = {}

def load_mnist():
    if 'mnist' not in _BASE:
        tr = torchvision.datasets.MNIST(DATA_ROOT, train=True, download=True)
        te = torchvision.datasets.MNIST(DATA_ROOT, train=False, download=True)
        f = lambda d: d.float().div(255).sub(0.1307).div(0.3081).reshape(-1, 784)
        _BASE['mnist'] = (f(tr.data).to(DEV), tr.targets.long().to(DEV), f(te.data).to(DEV), te.targets.long().to(DEV))
    return _BASE['mnist']

def load_cifar100():
    if 'cifar' not in _BASE:
        tr = torchvision.datasets.CIFAR100(DATA_ROOT, train=True, download=True)
        te = torchvision.datasets.CIFAR100(DATA_ROOT, train=False, download=True)
        mean = torch.tensor([0.5071, 0.4865, 0.4409]).view(1,3,1,1); std = torch.tensor([0.2673, 0.2564, 0.2762]).view(1,3,1,1)
        def prep(d):
            x = torch.from_numpy(d).float().div(255).permute(0,3,1,2)
            return ((x - mean) / std).reshape(len(x), -1)
        _BASE['cifar'] = (prep(tr.data).to(DEV), torch.tensor(tr.targets).long().to(DEV),
                          prep(te.data).to(DEV), torch.tensor(te.targets).long().to(DEV))
    return _BASE['cifar']

class Bench:
    """Task sequence. pmnist: single-head domain-incremental. cifar: multi-head task-incremental."""
    def __init__(self, kind, n_tasks, seed):
        self.kind, self.n_tasks, self.seed = kind, n_tasks, seed
        g = torch.Generator().manual_seed(1000 + seed)
        if kind == 'pmnist':
            Xtr, Ytr, Xte, Yte = load_mnist()
            self.in_dim, self.n_cls, self.multihead = 784, 10, False
            idx = torch.randperm(len(Xtr), generator=g)[:P['pm_ntrain']].to(DEV)
            self.Xtr, self.Ytr = Xtr[idx], Ytr[idx]
            idt = torch.randperm(len(Xte), generator=g)[:2000].to(DEV)
            self.Xte, self.Yte = Xte[idt], Yte[idt]
            perms = [torch.arange(784)] + [torch.randperm(784, generator=g) for _ in range(n_tasks - 1)]
            self.perms = [p.to(DEV) for p in perms]
            self.epochs, self.bs = P['pm_epochs'], 256
        elif kind == 'cifar':
            Xtr, Ytr, Xte, Yte = load_cifar100()
            self.in_dim, self.multihead = 3072, True
            self.n_cls = 100 // n_tasks
            classes = torch.randperm(100, generator=g).to(DEV)
            self.tr, self.te = [], []
            for t in range(n_tasks):
                cs = classes[t*self.n_cls:(t+1)*self.n_cls]
                lut = torch.full((100,), -1, dtype=torch.long, device=DEV); lut[cs] = torch.arange(self.n_cls, device=DEV)
                m1 = torch.isin(Ytr, cs); m2 = torch.isin(Yte, cs)
                self.tr.append((Xtr[m1], lut[Ytr[m1]])); self.te.append((Xte[m2], lut[Yte[m2]]))
            self.epochs, self.bs = P['cifar_epochs'], 128
        else:
            raise ValueError(kind)
    def train(self, t):
        if self.kind == 'pmnist': return self.Xtr[:, self.perms[t]], self.Ytr
        return self.tr[t]
    def test(self, t):
        if self.kind == 'pmnist': return self.Xte[:, self.perms[t]], self.Yte
        return self.te[t]

_BENCH_CACHE = {}
def get_bench(kind, n_tasks, seed):
    key = (kind, n_tasks, seed)
    if key not in _BENCH_CACHE:
        if len(_BENCH_CACHE) >= 3: _BENCH_CACHE.pop(next(iter(_BENCH_CACHE)))
        _BENCH_CACHE[key] = Bench(kind, n_tasks, seed)
    return _BENCH_CACHE[key]

load_mnist(); load_cifar100()
print("data ready")


In [ ]:
#@title 3 · Linear-algebra utilities (energy basis, null space, Cayley, hard-concrete)
BETA, GAMMA, ZETA = 2/3, -0.1, 1.1      # hard-concrete constants (Louizos et al.)

def orthonormalize(U):
    if U.shape[1] == 0: return U
    Q, R = torch.linalg.qr(U)
    s = torch.sign(torch.diagonal(R)); s[s == 0] = 1
    return Q * s

def top_eig(X, k=None):
    """X: n×d. Returns (eigenvalues of XᵀX descending, right-singular vectors V d×m). Dual trick if n<d."""
    X = X.to(F64); n, d = X.shape
    if n < d:
        ev, Pm = torch.linalg.eigh(X @ X.t()); ev = ev.flip(0); Pm = Pm.flip(1)
        keep = ev > ev[0].clamp_min(1e-30) * 1e-12
        ev, Pm = ev[keep], Pm[:, keep]
        V = (X.t() @ Pm) / ev.sqrt().unsqueeze(0)
    else:
        ev, V = torch.linalg.eigh(X.t() @ X); ev = ev.flip(0); V = V.flip(1)
        keep = ev > ev[0].clamp_min(1e-30) * 1e-12
        ev, V = ev[keep], V[:, keep]
    if k is not None: ev, V = ev[:k], V[:, :k]
    return ev, V

def energy_basis(X, U, eps):
    """GPM energy rule. Returns new orthonormal directions V (⟂U) such that (|UUᵀX|²+Σσ²)/|X|² >= eps."""
    X = X.to(F64); tot = (X ** 2).sum()
    if U.shape[1] > 0:
        XU = X @ U; proj = (XU ** 2).sum(); Xh = X - XU @ U.t()
    else:
        proj = torch.zeros((), dtype=F64, device=X.device); Xh = X
    ev, V = top_eig(Xh)
    if tot <= 0 or ev.numel() == 0 or proj / tot >= eps: return V[:, :0], ev[:0]
    cum = proj + torch.cumsum(ev, 0)
    ok = (cum / tot >= eps).nonzero()
    k = int(ok[0].item()) + 1 if ok.numel() > 0 else ev.numel()
    return V[:, :k], ev[:k]

def append_dirs(U, V, tol=1e-3):
    """Append the part of span(V) that is not already in span(U)."""
    if V.shape[1] == 0: return U
    Vr = V - U @ (U.t() @ V) if U.shape[1] > 0 else V
    ev, W = top_eig(Vr.t())
    keep = ev > tol ** 2
    if keep.sum() == 0: return U
    W = W[:, keep]
    return orthonormalize(torch.cat([U, W], 1)) if U.shape[1] > 0 else orthonormalize(W)

def null_basis(U, d):
    r = U.shape[1]
    if r == 0: return torch.eye(d, dtype=F64, device=DEV)
    Qc, _ = torch.linalg.qr(U, mode='complete')
    return Qc[:, r:]

def fill_orth(V, p, U=None):
    """Pad V (d×k) with random orthonormal columns (⟂ V and ⟂ U) until it has p columns."""
    d, k = V.shape
    if k >= p: return V[:, :p]
    R = torch.randn(d, p - k, dtype=F64, device=DEV)
    basis = V if (U is None or U.shape[1] == 0) else torch.cat([U, V], 1)
    if basis.shape[1] > 0: R = R - basis @ (basis.t() @ R)
    return torch.cat([V, orthonormalize(R)], 1)

def cayley_step(B, G, tau):
    """Wen–Yin Cayley retraction on the Stiefel manifold: B+ = (I+τ/2 A)^-1 (I-τ/2 A) B, A = G Bᵀ - B Gᵀ."""
    Um = torch.cat([G, B], 1); Vm = torch.cat([B, -G], 1)
    M = torch.eye(Um.shape[1], device=B.device, dtype=B.dtype) + (tau / 2) * (Vm.t() @ Um)
    return B - tau * Um @ torch.linalg.solve(M, Vm.t() @ B)

def clean(o):
    """JSON-safe conversion (NaN/inf -> None)."""
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, float)):
        o = float(o); return None if (math.isnan(o) or math.isinf(o)) else o
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if torch.is_tensor(o): return clean(o.item() if o.numel() == 1 else o.tolist())
    return o


In [ ]:
#@title 4 · Model, adapter and Learner base class
class Adapter(nn.Module):
    """ΔW = S · diag(z) · Qᵀ   with Q: d_in×p, S: d_out×p.
    qmode: 'hard' (Q = N·B, B on Stiefel, Riemannian/Cayley), 'soft' (Q raw, effective Q=(I-UΛUᵀ)Q),
           'free' (Q raw, unconstrained), 'fixed' (Q buffer)."""
    def __init__(self, d_in, d_out, p, qmode, gmode, N=None, U=None, Lam=None, B0=None, Q0=None, zfix=None, la_init=3.0, s_std=0.02):
        super().__init__()
        self.p, self.qmode, self.gmode = p, qmode, gmode
        self.S = nn.Parameter(torch.randn(d_out, p, device=DEV) * s_std)
        self.B = self.Qraw = None
        if qmode == 'hard':
            self.register_buffer('N', N.float().contiguous()); self.B = nn.Parameter(B0.float().clone())
        elif qmode == 'soft':
            self.Qraw = nn.Parameter(Q0.float().clone())
            self.register_buffer('U', U.float().contiguous()); self.register_buffer('Lam', Lam.float().contiguous())
        elif qmode == 'free':
            self.Qraw = nn.Parameter(Q0.float().clone())
        else:
            self.register_buffer('Qfix', Q0.float().clone())
        self.logit = None
        if gmode == 'hc': self.logit = nn.Parameter(torch.full((p,), la_init, device=DEV))
        elif gmode == 'random_mask': self.register_buffer('zfix', zfix.float().clone())
    def q_eff(self):
        if self.qmode == 'hard': return self.N @ self.B
        if self.qmode == 'soft': return self.Qraw - self.U @ (self.Lam[:, None] * (self.U.t() @ self.Qraw))
        if self.qmode == 'free': return self.Qraw
        return self.Qfix
    def gate(self):
        if self.gmode == 'none': return None
        if self.gmode == 'random_mask': return self.zfix
        if self.training:
            u = torch.rand_like(self.logit).clamp(1e-6, 1 - 1e-6)
            s = torch.sigmoid((torch.log(u) - torch.log(1 - u) + self.logit) / BETA)
        else:
            s = torch.sigmoid(self.logit)
        return (s * (ZETA - GAMMA) + GAMMA).clamp(0, 1)
    def l0(self):
        return torch.sigmoid(self.logit - BETA * math.log(-GAMMA / ZETA)).mean()
    def forward(self, x):
        h = x @ self.q_eff(); z = self.gate()
        if z is not None: h = h * z
        return h @ self.S.t()
    @torch.no_grad()
    def fold(self):
        self.eval()
        Q = self.q_eff(); z = self.gate()
        if z is None: z = torch.ones(self.p, device=Q.device)
        keep = z > 0
        Sk = self.S[:, keep] * z[keep]; Qk = Q[:, keep]
        return Sk @ Qk.t(), Qk, int(keep.sum().item()), z

class SharedLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        w = torch.empty(d_out, d_in, device=DEV); nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        self.weight = nn.Parameter(w); self.ad = None; self.owner = None
    def forward(self, x, t=0):
        W = self.weight
        if self.owner is not None: W = W * (self.owner <= t)
        y = F.linear(x, W)
        if self.ad is not None: y = y + self.ad(x)
        return y

class Net(nn.Module):
    def __init__(self, in_dim, width, n_cls, multihead, n_tasks):
        super().__init__()
        dims = [in_dim, width, width] if multihead else [in_dim, width, width, n_cls]
        self.layers = nn.ModuleList([SharedLayer(dims[i], dims[i+1]) for i in range(len(dims) - 1)])
        self.multihead = multihead
        self.heads = nn.ModuleList([nn.Linear(dims[-1], n_cls) for _ in range(n_tasks)]) if multihead else None
        self.to(DEV)
    def forward(self, x, t=0, ret_aux=False):
        ins, pres = [], []; h = x; L = len(self.layers)
        for i, lay in enumerate(self.layers):
            ins.append(h); z = lay(h, t); pres.append(z)
            h = z if (i == L - 1 and not self.multihead) else F.relu(z)
        out = self.heads[t](h) if self.multihead else h
        return (out, ins, pres) if ret_aux else out

class Learner:
    track_old = True
    def __init__(self, spec, bench):
        self.spec, self.hp, self.bench = spec, spec['hp'], bench
        self.N, self.width = bench.n_tasks, spec['width']
        self.new_net()
        self.L = len(self.net.layers)
        self.gen = torch.Generator(device=DEV).manual_seed(spec['seed'] * 7919 + 17)
        self.events = []; self.snap = None; self.W_before = None
        self.has_bound = False
    def new_net(self):
        self.net = Net(self.bench.in_dim, self.width, self.bench.n_cls, self.bench.multihead, self.N)
    def head_params(self, t):
        return list(self.net.heads[t].parameters()) if self.net.multihead else []
    # ---------------- evaluation ----------------
    @torch.no_grad()
    def evaluate(self, s):
        self.net.eval(); X, Y = self.bench.test(s)
        lg = self.net(X, s)
        return (lg.argmax(1) == Y).float().mean().item(), F.cross_entropy(lg, Y).item()
    def eval_old(self, t):
        r = [self.evaluate(s) for s in range(t)]
        return np.array([a for a, _ in r]), np.array([l for _, l in r])
    @torch.no_grad()
    def collect_ins(self, t, n=None):
        X, _ = self.bench.train(t); n = min(n or P['n_R'], X.shape[0])
        idx = torch.randperm(X.shape[0], device=DEV, generator=self.gen)[:n]
        self.net.eval(); _, ins, _ = self.net(X[idx], t, ret_aux=True)
        return [h.detach().to(F64) for h in ins]
    # ---------------- generic training loop ----------------
    def fit(self, t, groups, epochs=None, extra_loss=None, post_backward=None, pre_step=None, post_step=None, need_aux=False, replay=None):
        groups = [g for g in groups if len(g['params']) > 0]
        X, Y = self.bench.train(t); n = X.shape[0]; bs = self.bench.bs
        epochs = epochs or self.bench.epochs
        opt = torch.optim.Adam(groups) if groups else None
        total = epochs * math.ceil(n / bs); step = 0; loss = torch.zeros(())
        self.net.train()
        for ep in range(epochs):
            perm = torch.randperm(n, device=DEV, generator=self.gen)
            for i in range(0, n, bs):
                idx = perm[i:i+bs]; xb, yb = X[idx], Y[idx]; ins = None
                if need_aux: out, ins, _ = self.net(xb, t, ret_aux=True)
                else: out = self.net(xb, t)
                loss = F.cross_entropy(out, yb)
                if extra_loss is not None: loss = loss + extra_loss(step, total, ins)
                if replay is not None: loss = loss + replay(step)
                if opt is None: step += 1; continue
                opt.zero_grad(set_to_none=True); loss.backward()
                if post_backward: post_backward(step, total)
                if pre_step: pre_step()
                opt.step()
                if post_step: post_step(step, total)
                step += 1
        self.net.eval()
        return float(loss.item())
    # ---------------- hooks ----------------
    def pre_task(self, t):
        self.W_before = [l.weight.detach().clone() for l in self.net.layers]
        self.snap = None
        if t == 0 or not self.track_old: return
        self.net.eval(); lg = {}; eps = [0.0] * self.L; Ul = getattr(self, 'U', None)
        with torch.no_grad():
            for s in range(t):
                X, _ = self.bench.test(s)
                out, ins, _ = self.net(X, s, ret_aux=True); lg[s] = out
                if Ul is not None:
                    for l, h in enumerate(ins):
                        U = Ul[l]; hh = h.double()
                        res = hh - (hh @ U) @ U.t() if U.shape[1] > 0 else hh
                        eps[l] = max(eps[l], res.norm(dim=1).max().item())
        self.snap = (lg, eps)
    def train_task(self, t): raise NotImplementedError
    def post_task(self, t): pass
    def diag(self, t): return {}
    @torch.no_grad()
    def drift_stats(self, t):
        if t == 0 or self.snap is None: return {}
        lg, eps = self.snap; mx, sm, cnt = 0.0, 0.0, 0
        for s in range(t):
            X, _ = self.bench.test(s); d = (self.net(X, s) - lg[s]).norm(dim=1)
            mx = max(mx, d.max().item()); sm += d.sum().item(); cnt += d.numel()
        row = dict(drift_max=mx, drift_mean=sm / cnt, thm2_valid=bool(self.has_bound))
        if self.has_bound and getattr(self, 'U', None) is not None:
            Wn = [l.weight.detach() for l in self.net.layers]
            opn = [torch.linalg.matrix_norm(W, 2).item() for W in Wn]
            dW = [torch.linalg.matrix_norm(W - Wb, 2).item() for W, Wb in zip(Wn, self.W_before)]
            b = 0.0
            for l in range(self.L):
                b += float(np.prod(opn[l+1:])) * dW[l] * eps[l]
            if self.net.multihead:
                b *= max(torch.linalg.matrix_norm(self.net.heads[s].weight.detach(), 2).item() for s in range(t))
            row['thm2_bound'] = b
        return row


In [ ]:
#@title 5 · Baselines: FT, STL, ER, Parseval, OGD, PackNet, GPM / SGP / random-projector
LR = 1e-3

class FT(Learner):
    def train_task(self, t):
        ws = [l.weight for l in self.net.layers]
        for w in ws: w.requires_grad_(True)
        self.fit(t, [dict(params=ws + self.head_params(t), lr=self.hp.get('lr', LR))])

class STL(FT):
    """Single-task learning from scratch (reference for plasticity). Old tasks are NOT tracked."""
    track_old = False
    def train_task(self, t):
        self.new_net(); super().train_task(t)
    def eval_old(self, t): return np.full(t, np.nan), np.full(t, np.nan)
    def evaluate(self, s):
        return super().evaluate(s)

class Parseval(FT):
    def train_task(self, t):
        lam = self.hp.get('lam', 0.01)
        def extra(step, total, ins):
            pen = 0.0
            for l in self.net.layers:
                W = l.weight; G = W @ W.t() if W.shape[0] <= W.shape[1] else W.t() @ W
                pen = pen + ((G - torch.eye(G.shape[0], device=DEV)) ** 2).mean()
            return lam * pen
        ws = [l.weight for l in self.net.layers]
        for w in ws: w.requires_grad_(True)
        self.fit(t, [dict(params=ws + self.head_params(t), lr=LR)], extra_loss=extra)

class ER(Learner):
    def __init__(self, spec, bench):
        super().__init__(spec, bench); self.buf = {}; self.rng = random.Random(spec['seed'])
    def train_task(self, t):
        nb = self.hp.get('n_buf', 200); bs = self.bench.bs
        def replay(step):
            if not self.buf: return 0.0
            tot = 0.0; ks = self.rng.sample(list(self.buf), min(2, len(self.buf)))
            for s in ks:
                xs, ys = self.buf[s]; idx = torch.randint(0, xs.shape[0], (min(bs // 2, xs.shape[0]),), device=DEV)
                tot = tot + F.cross_entropy(self.net(xs[idx], s), ys[idx])
            return tot / len(ks)
        ws = [l.weight for l in self.net.layers]
        for w in ws: w.requires_grad_(True)
        self.fit(t, [dict(params=ws + self.head_params(t), lr=LR)], replay=replay)
        X, Y = self.bench.train(t); idx = torch.randperm(X.shape[0], device=DEV, generator=self.gen)[:nb]
        self.buf[t] = (X[idx].clone(), Y[idx].clone())
    def diag(self, t): return dict(extra_mem_floats=sum(v[0].numel() for v in self.buf.values()))

class OGD(Learner):
    """Orthogonal Gradient Descent (step-projected, Adam) with a small per-task gradient memory."""
    def __init__(self, spec, bench):
        super().__init__(spec, bench); self.Bq = None
    def flat(self): return torch.cat([l.weight.reshape(-1) for l in self.net.layers])
    @torch.no_grad()
    def set_flat(self, v):
        i = 0
        for l in self.net.layers:
            n = l.weight.numel(); l.weight.copy_(v[i:i+n].view_as(l.weight)); i += n
    def train_task(self, t):
        ws = [l.weight for l in self.net.layers]
        for w in ws: w.requires_grad_(True)
        st = {}
        def pre(): st['old'] = self.flat().detach().clone()
        @torch.no_grad()
        def post(step, total):
            if self.Bq is None: return
            d = self.flat() - st['old']; d = d - self.Bq @ (self.Bq.t() @ d)
            self.set_flat(st['old'] + d)
        self.fit(t, [dict(params=ws + self.head_params(t), lr=LR)], pre_step=pre, post_step=post)
    def post_task(self, t):
        ng = self.hp.get('n_g', 20); X, _ = self.bench.train(t); self.net.eval()
        idx = torch.randperm(X.shape[0], device=DEV, generator=self.gen)[:ng]
        ws = [l.weight for l in self.net.layers]; new = []
        for i in idx:
            out = self.net(X[i:i+1], t); g = torch.autograd.grad(out.max(), ws)
            v = torch.cat([x.reshape(-1) for x in g]).detach(); n0 = v.norm()
            if n0 < 1e-12: continue
            for _ in range(2):
                if self.Bq is not None: v = v - self.Bq @ (self.Bq.t() @ v)
                if new: 
                    Bn = torch.stack(new, 1); v = v - Bn @ (Bn.t() @ v)
            if v.norm() > 1e-3 * n0: new.append(v / v.norm())
        if new:
            Bn = torch.stack(new, 1); self.Bq = Bn if self.Bq is None else torch.cat([self.Bq, Bn], 1)
    def diag(self, t): return dict(extra_mem_floats=0 if self.Bq is None else self.Bq.numel(), mem_dirs=0 if self.Bq is None else self.Bq.shape[1])

class PackNet(Learner):
    """PackNet-lite: iterative magnitude pruning, equal share per task, weights owned by task t masked for tasks < t."""
    BIG = 10 ** 6
    def __init__(self, spec, bench):
        super().__init__(spec, bench)
        for l in self.net.layers: l.owner = torch.full(l.weight.shape, self.BIG, dtype=torch.int32, device=DEV)
    def train_task(self, t):
        layers = self.net.layers; BIG = self.BIG
        for l in layers:
            l.weight.requires_grad_(True); l.free = (l.owner == BIG)
            l.owner_saved = l.owner.clone(); l.owner = torch.where(l.free, torch.full_like(l.owner, t), l.owner)
        def pb(step, total):
            for l in layers:
                if l.weight.grad is not None: l.weight.grad.mul_(l.free)
        groups = [dict(params=[l.weight for l in layers] + self.head_params(t), lr=LR)]
        self.fit(t, groups, post_backward=pb)
        with torch.no_grad():
            for l in layers:
                nfree = int(l.free.sum().item())
                if t < self.N - 1 and nfree > 0:
                    keep_n = max(1, int(round(nfree / (self.N - t))))
                    w = l.weight.abs()
                    thr = torch.topk(w[l.free], keep_n).values.min()
                    newown = l.free & (w >= thr)
                    l.owner = torch.where(newown, torch.full_like(l.owner, t), l.owner_saved)
                    l.weight.mul_(~(l.free & ~newown))
                else:
                    l.owner = torch.where(l.free, torch.full_like(l.owner, t), l.owner_saved)
                l.free = (l.owner == t)
        self.fit(t, groups, epochs=max(1, self.bench.epochs // 3), post_backward=pb)
    def diag(self, t):
        d = {}
        for i, l in enumerate(self.net.layers):
            d[f'owned_frac_l{i}'] = float((l.owner < self.BIG).float().mean().item())
        d['extra_mem_floats'] = sum(l.owner.numel() for l in self.net.layers)
        return d

class GPM(Learner):
    """variant: 'gpm' (hard projection), 'sgp' (scaled/soft projection), 'rand' (random projector of GPM's rank)."""
    def __init__(self, spec, bench):
        super().__init__(spec, bench)
        self.variant = self.hp.get('variant', 'gpm'); self.eps = self.hp.get('eps', 0.97); self.alpha = self.hp.get('alpha', 3.0)
        self.U = [torch.zeros(l.d_in, 0, dtype=F64, device=DEV) for l in self.net.layers]
        self.Lam = [torch.zeros(0, dtype=F64, device=DEV) for _ in self.net.layers]
        self.footprint = [[] for _ in self.net.layers]
        self.has_bound = self.variant == 'gpm'
    def train_task(self, t):
        layers = self.net.layers
        for l in layers: l.weight.requires_grad_(True)
        Uf = [U.float() for U in self.U]; Lf = [L.float() for L in self.Lam]; st = {}
        def pre(): st['old'] = [l.weight.detach().clone() for l in layers]
        @torch.no_grad()
        def post(step, total):
            for l, old, U, Lm in zip(layers, st['old'], Uf, Lf):
                if U.shape[1] == 0: continue
                dW = l.weight - old; dW = dW - ((dW @ U) * Lm[None, :]) @ U.t(); l.weight.copy_(old + dW)
        self.fit(t, [dict(params=[l.weight for l in layers] + self.head_params(t), lr=LR)], pre_step=pre, post_step=post)
    def post_task(self, t):
        ins = self.collect_ins(t)
        for l, X in enumerate(ins):
            V, ev = energy_basis(X, self.U[l], self.eps); k = V.shape[1]
            if k > 0:
                if self.variant == 'rand': V = fill_orth(V[:, :0], k, self.U[l])
                if self.variant == 'sgp':
                    s = ev.sqrt(); lam = self.alpha * s / ((self.alpha - 1) * s + s.max())
                else: lam = torch.ones(k, dtype=F64, device=DEV)
                self.U[l] = orthonormalize(torch.cat([self.U[l], V], 1)); self.Lam[l] = torch.cat([self.Lam[l], lam])
            self.footprint[l].append(k)
    def diag(self, t):
        d = {}; mem = 0
        for l, lay in enumerate(self.net.layers):
            r = self.U[l].shape[1]; D = lay.d_in
            d[f'U_rank_l{l}'] = r; d[f'free_l{l}'] = D - r; d[f'rho_l{l}'] = r / D; d[f'foot_l{l}'] = self.footprint[l][-1]; mem += self.U[l].numel()
        d['extra_mem_floats'] = mem
        return d


In [ ]:
#@title 6 · SOCR learner (sparse adapters in a learned orthogonal frame + capacity recycling)
SOCR_DEF = dict(
    ortho='hard',        # 'hard' (Stiefel + null space) | 'soft' (penalty + projected Q) | 'off' | 'olora' (weight-space orthogonality penalty, O-LoRA-style)
    frame='learned',     # 'learned' | 'fixed_data' | 'fixed_random' | 'fixed_gauss'
    gate='hc',           # 'hc' (hard-concrete L0) | 'none' | 'random_mask'
    lam0=0.05, lam_r=0.05, lam_o=1.0, rand_keep=0.5,
    reserve='input',     # 'input' (Option A) | 'input+write' (Option B)
    recycle='none',      # 'none' | 'R2' | 'R2R3' | 'overlap'
    eps=0.97, p=None, p_min=4,
    lr=3e-3, lr_gate=3e-2, lr_q=1e-3, lr_B=0.02, lr_head=1e-3, lr_full=1e-3,
    base_init='trained_task0', share_first=False, shuffle_assign=False,
    free_trig=0.5, free_target=1.0, r2_rel=0.02, kappa=0.25, repair_steps=100,
    nuc_every=2, nuc_n=64)

class SOCR(Learner):
    def __init__(self, spec, bench):
        super().__init__(spec, bench)
        c = self.c = {**SOCR_DEF, **spec['hp']}
        self.dims = [l.d_in for l in self.net.layers]
        self.p = c['p'] or max(8, self.width // 4)
        for l in self.net.layers: l.weight.requires_grad_(False)
        self.U = [torch.zeros(d, 0, dtype=F64, device=DEV) for d in self.dims]
        self.Qh = [[] for _ in self.dims]; self.footprint = [[] for _ in self.dims]
        self.track_A = c['recycle'] in ('R2', 'R2R3', 'overlap'); self.track_G = c['recycle'] in ('R2', 'R2R3')
        self.A = [torch.zeros(d, d, dtype=F64, device=DEV) if self.track_A else None for d in self.dims]
        self.G = [torch.zeros(l.d_out, l.d_out, dtype=F64, device=DEV) if self.track_G else None for l in self.net.layers]
        self.pseudo = {}; self.noadapt = {0} if c['share_first'] else set()
        self.has_bound = c['ortho'] in ('hard', 'soft') and c['recycle'] != 'overlap'
        self.released_now = 0; self.ad_stats = {}; self.last_Q = {}

    def src_task(self, t):
        if not self.c['shuffle_assign'] or self.N < 2: return t
        return (t + 1 + self.spec['seed'] % (self.N - 1)) % self.N

    # ------------------------------------------------------------------ training
    def train_task(self, t):
        c = self.c; self.released_now = 0; self.last_Q = {}; self.ad_stats = {}
        if t > 0 and c['recycle'] in ('R2', 'R2R3'): self.release_stage(t)
        if t == 0 and c['base_init'] == 'trained_task0':
            ws = [l.weight for l in self.net.layers]
            for w in ws: w.requires_grad_(True)
            self.fit(t, [dict(params=ws + self.head_params(t), lr=c['lr_full'])])
            for w in ws: w.requires_grad_(False)
        else:
            self.train_adapters(t)

    def importance_lambda(self, l):
        U = self.U[l]; e = ((self.A[l] @ U) * U).sum(0)
        med = e.median().clamp_min(1e-12)
        return e / (e + self.c['kappa'] * med)

    def train_adapters(self, t):
        c = self.c; net = self.net; ins = self.collect_ins(self.src_task(t)); ads = {}
        for l, lay in enumerate(net.layers):
            if l in self.noadapt: continue
            d, m = lay.d_in, lay.d_out; U = self.U[l]; r = U.shape[1]; f = d - r; Xl = ins[l]
            use_null = c['ortho'] in ('hard', 'soft')
            overlap = use_null and c['recycle'] == 'overlap' and f < max(c['p_min'], self.p * c['free_trig'])
            Nb = Lam = B0 = None
            if not use_null:
                p_l = self.p
                if c['ortho'] == 'olora' or c['frame'] == 'fixed_gauss':
                    Q0 = torch.randn(d, p_l, dtype=F64, device=DEV) / math.sqrt(d)
                elif c['frame'] == 'fixed_random':
                    Q0 = fill_orth(torch.zeros(d, 0, dtype=F64, device=DEV), p_l)
                else:
                    ev, V = top_eig(Xl, p_l); Q0 = fill_orth(V, p_l)
                qmode = 'free' if (c['ortho'] == 'olora' or c['frame'] == 'learned') else 'fixed'
            elif overlap:
                p_l = self.p; Lam = self.importance_lambda(l)
                Xh = Xl - ((Xl @ U) * Lam[None, :]) @ U.t()
                ev, V = top_eig(Xh, p_l); Q0 = fill_orth(V, p_l); qmode = 'soft'
            else:
                if f < c['p_min']:
                    self.ad_stats[l] = dict(starved=1); continue
                p_l = min(self.p, f); Nb = null_basis(U, d)
                if c['frame'] in ('learned', 'fixed_data'):
                    Xh = Xl - (Xl @ U) @ U.t() if r > 0 else Xl
                    ev, V = top_eig(Xh, p_l); V = fill_orth(V, p_l, U)
                else:
                    V = fill_orth(torch.zeros(d, 0, dtype=F64, device=DEV), p_l, U)
                B0 = orthonormalize(Nb.t() @ V); Q0 = Nb @ B0
                if c['ortho'] == 'hard': qmode = 'hard' if c['frame'] == 'learned' else 'fixed'
                else:
                    qmode = 'soft' if c['frame'] == 'learned' else 'fixed'; Lam = torch.ones(r, dtype=F64, device=DEV)
            zfix = (torch.rand(p_l, device=DEV) < c['rand_keep']).float() if c['gate'] == 'random_mask' else None
            ads[l] = Adapter(d, m, p_l, qmode, c['gate'], N=Nb, U=U, Lam=Lam, B0=B0, Q0=Q0, zfix=zfix)
            lay.ad = ads[l]
        groups = [dict(params=[a.S for a in ads.values()], lr=c['lr']),
                  dict(params=[a.logit for a in ads.values() if a.logit is not None], lr=c['lr_gate']),
                  dict(params=[a.Qraw for a in ads.values() if a.Qraw is not None], lr=c['lr_q']),
                  dict(params=self.head_params(t), lr=c['lr_head'])]
        Bs = [a.B for a in ads.values() if a.qmode == 'hard']
        I_ = {}
        def extra(step, total, ins_b):
            warm = min(1.0, step / (0.3 * total)); loss = 0.0
            if c['gate'] == 'hc' and c['lam0'] > 0 and ads:
                loss = loss + warm * c['lam0'] * sum(a.l0() for a in ads.values())
            if c['lam_r'] > 0 and ins_b is not None and step % c['nuc_every'] == 0:
                for l in range(1, self.L):
                    X = ins_b[l][:c['nuc_n']]
                    loss = loss + warm * c['lam_r'] * c['nuc_every'] * torch.linalg.matrix_norm(X, ord='nuc') / X.shape[0]
            if c['ortho'] == 'soft':
                for a in ads.values():
                    Qe = a.q_eff(); I = I_.setdefault(a.p, torch.eye(a.p, device=DEV))
                    loss = loss + c['lam_o'] * ((Qe.t() @ Qe - I) ** 2).sum()
            if c['ortho'] == 'olora':
                for l, a in ads.items():
                    if self.Qh[l]:
                        Qp = torch.cat(self.Qh[l], 1); loss = loss + c['lam_o'] * ((Qp.t() @ a.q_eff()) ** 2).sum()
            return loss
        def post_backward(step, total):
            with torch.no_grad():
                for B in Bs:
                    if B.grad is None: continue
                    gn = B.grad.norm()
                    if gn > 1e-12: B.copy_(cayley_step(B, B.grad / gn, c['lr_B']))
                    B.grad = None
                    if (step + 1) % 50 == 0: B.copy_(orthonormalize(B.double()).float())
        has_params = any(len(g['params']) > 0 for g in groups)
        if has_params:
            self.fit(t, groups, extra_loss=extra, post_backward=post_backward, need_aux=(c['lam_r'] > 0))
        # ---- fold adapters into the frozen weights
        for l, a in ads.items():
            lay = net.layers[l]; U = self.U[l]
            dW, Qk, nk, z = a.fold()
            leak = ((dW.double() @ U).norm() / (dW.double().norm() + 1e-12)).item() if U.shape[1] > 0 else 0.0
            oe = (Qk.t() @ Qk - torch.eye(Qk.shape[1], device=DEV)).norm().item() if Qk.shape[1] > 0 else 0.0
            qc = 0.0
            for Qs in self.Qh[l]:
                if Qk.shape[1] > 0: qc = max(qc, torch.linalg.matrix_norm(Qs.t() @ Qk, 2).item())
            with torch.no_grad(): lay.weight.add_(dW)
            lay.ad = None
            self.ad_stats[l] = dict(p=a.p, kept=nk, sparsity=1 - nk / a.p, leak=leak, orth_err=oe, qcos=qc, starved=0, overlap=int(a.qmode == 'soft' and c['recycle'] == 'overlap'))
            if nk > 0:
                self.Qh[l].append(Qk.detach().clone())
                if c['ortho'] in ('hard', 'soft'): self.last_Q[l] = Qk.detach().clone()
        for l, lay in enumerate(net.layers): lay.ad = None

    # ------------------------------------------------------------------ freezing
    def post_task(self, t):
        c = self.c; src = self.src_task(t); ins = self.collect_ins(src)
        for l, X in enumerate(ins):
            if l in self.noadapt: continue
            V, ev = energy_basis(X, self.U[l], c['eps']); k = V.shape[1]; U = self.U[l]
            if k > 0: U = orthonormalize(torch.cat([U, V], 1))
            if c['reserve'] == 'input+write' and l in self.last_Q: U = append_dirs(U, self.last_Q[l].double())
            self.U[l] = U; self.footprint[l].append(k)
            if self.track_A: self.A[l] += X.t() @ X / X.shape[0]
        if self.track_G: self.accumulate_G(src)
        if c['recycle'] == 'R2R3':
            X0 = ins[0].float(); mu = X0.mean(0); Xc = X0 - mu; q = max(1, min(64, Xc.shape[0] - 1, Xc.shape[1]))
            _, S, V = torch.svd_lowrank(Xc, q=q, niter=2)
            self.pseudo[src] = (mu, V * (S / math.sqrt(Xc.shape[0]))[None, :])

    def accumulate_G(self, t):
        X, Y = self.bench.train(t); n = min(P['n_R'], X.shape[0])
        idx = torch.randperm(X.shape[0], device=DEV, generator=self.gen)[:n]
        self.net.eval(); xb = X[idx].clone().requires_grad_(True)
        out, ins, pres = self.net(xb, t, ret_aux=True)
        loss = F.cross_entropy(out, Y[idx], reduction='sum')
        gs = torch.autograd.grad(loss, pres)
        for l, g in enumerate(gs):
            g = g.detach().double(); self.G[l] += g.t() @ g / n

    # ------------------------------------------------------------------ recycling
    @torch.no_grad()
    def apply_release(self, l, sel):
        U = self.U[l]; Ur = U[:, sel].float(); W = self.net.layers[l].weight
        W.copy_(W - (W @ Ur) @ Ur.t())
        mask = torch.ones(U.shape[1], dtype=torch.bool, device=DEV); mask[sel] = False
        self.U[l] = U[:, mask]; self.released_now += int(len(sel))

    @torch.no_grad()
    def iota(self, l):
        U = self.U[l]; WU = self.net.layers[l].weight.detach().double() @ U
        a = ((self.A[l] @ U) * U).sum(0); g = ((self.G[l] @ WU) * WU).sum(0)
        return 0.5 * a * g, a

    def release_stage(self, t):
        c = self.c; layers = self.net.layers; need = {}
        for l, lay in enumerate(layers):
            if l in self.noadapt: continue
            d = lay.d_in; r = self.U[l].shape[1]; f = d - r
            if r > 0 and f < max(c['p_min'], self.p * c['free_trig']):
                n = min(r, max(0, min(d, int(round(self.p * c['free_target']))) - f))
                if n > 0: need[l] = n
        if not need: return
        teacher = copy.deepcopy(self.net) if c['recycle'] == 'R2R3' else None
        acc0, loss0 = self.eval_old(t); sel2 = {}; pred = 0.0; cnt = {}
        for l, n in need.items():
            io, _ = self.iota(l); order = torch.argsort(io)
            if c['recycle'] == 'R2': k2 = n
            else: k2 = min(n, int((torch.cumsum(io[order], 0) <= c['r2_rel'] * io.sum()).sum().item()))
            if k2 > 0:
                sel2[l] = order[:k2]; pred += io[order[:k2]].clamp_min(0).sqrt().sum().item()
        for l, sel in sel2.items(): self.apply_release(l, sel); cnt[l] = int(len(sel))
        acc1, loss1 = self.eval_old(t)
        if cnt:
            self.events.append(dict(task=t, stage='R2', n_released=sum(cnt.values()), per_layer=cnt, n_requested=sum(need.values()),
                pred_bound=pred ** 2, meas_dloss_sum=float((loss1 - loss0).sum()), meas_dloss_mean=float((loss1 - loss0).mean()),
                acc_drop_mean=float((acc0 - acc1).mean())))
        if c['recycle'] != 'R2R3': return
        sel3 = {}; pred3 = 0.0; cnt3 = {}
        for l, n in need.items():
            d = layers[l].d_in; r = self.U[l].shape[1]; f = d - r
            n3 = min(r, max(0, min(d, int(round(self.p * c['free_target']))) - f))
            if n3 > 0:
                io, e = self.iota(l); sel = torch.argsort(e)[:n3]; sel3[l] = sel; pred3 += io[sel].clamp_min(0).sqrt().sum().item()
        if not sel3: return
        for l, sel in sel3.items(): self.apply_release(l, sel); cnt3[l] = int(len(sel))
        acc2, loss2 = self.eval_old(t)
        pre = float((loss2 - loss0).sum())
        if c['repair_steps'] > 0 and self.pseudo: self.repair(t, teacher)
        acc3, loss3 = self.eval_old(t)
        self.events.append(dict(task=t, stage='R3', n_released=sum(cnt3.values()), per_layer=cnt3, pred_bound=pred3 ** 2,
            meas_dloss_sum=float((loss2 - loss1).sum()), meas_dloss_mean=float((loss2 - loss1).mean()), acc_drop_mean=float((acc1 - acc2).mean()),
            total_dloss_prerepair=pre, total_dloss_postrepair=float((loss3 - loss0).sum()),
            acc_drop_total_prerepair=float((acc0 - acc2).mean()), acc_drop_total_postrepair=float((acc0 - acc3).mean())))

    def repair(self, t, teacher):
        c = self.c; layers = self.net.layers; old = [l.weight.detach().clone() for l in layers]
        Uf = [U.float() for U in self.U]
        for l in layers: l.weight.requires_grad_(True)
        opt = torch.optim.Adam([l.weight for l in layers], lr=1e-3); rng = random.Random(t)
        tasks = [s for s in range(t) if s in self.pseudo]; teacher.eval()
        for st in range(c['repair_steps']):
            loss = 0.0
            for s in rng.sample(tasks, min(8, len(tasks))):
                mu, Lf = self.pseudo[s]
                x = mu[None] + torch.randn(128, Lf.shape[1], device=DEV) @ Lf.t()
                with torch.no_grad(): tl = teacher(x, s)
                loss = loss + F.mse_loss(self.net(x, s), tl)
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                for l, o, U in zip(layers, old, Uf):
                    l.weight.copy_(o + ((l.weight - o) @ U) @ U.t() if U.shape[1] > 0 else o)
        for l in layers: l.weight.requires_grad_(False)
        self.net.eval()

    # ------------------------------------------------------------------ diagnostics
    def diag(self, t):
        d = dict(released_now=self.released_now); mem = 0; sp = []
        for l, lay in enumerate(self.net.layers):
            r = self.U[l].shape[1]; D = lay.d_in; mem += self.U[l].numel()
            d[f'U_rank_l{l}'] = r; d[f'free_l{l}'] = D - r; d[f'rho_l{l}'] = r / D
            d[f'foot_l{l}'] = self.footprint[l][-1] if self.footprint[l] else 0
            st = self.ad_stats.get(l)
            if st:
                for k, v in st.items(): d[f'{k}_l{l}'] = v
                if 'sparsity' in st: sp.append(st['sparsity'])
        d['sparsity_mean'] = float(np.mean(sp)) if sp else float('nan')
        for k in ('A', 'G'):
            for M in getattr(self, k):
                if M is not None: mem += M.numel()
        d['extra_mem_floats'] = mem
        return d


In [ ]:
#@title 7 · Runner, run registry and result storage
METHODS = dict(FT=FT, STL=STL, Parseval=Parseval, ER=ER, OGD=OGD, PackNet=PackNet, GPM=GPM, SOCR=SOCR)
F_TASK = os.path.join(OUT, "task_rows.jsonl"); F_META = os.path.join(OUT, "run_meta.jsonl")
F_EVT = os.path.join(OUT, "events.jsonl");     F_ERR = os.path.join(OUT, "errors.jsonl")

def cfg_key(s): return f"{s['bench']}|N{s['N']}|d{s['width']}|{s['method']}|{json.dumps(s['hp'], sort_keys=True)}"
def run_id(s):  return hashlib.md5((cfg_key(s) + f"|s{s['seed']}").encode()).hexdigest()[:12]
def append_jsonl(path, objs):
    with open(path, "a") as f:
        for o in objs: f.write(json.dumps(clean(o)) + "\n")
def read_jsonl(path):
    if not os.path.exists(path): return []
    out = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try: out.append(json.loads(line))
                except Exception: pass
    return out

def run_one(spec):
    seed = spec['seed']; rid = run_id(spec)
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    bench = get_bench(spec['bench'], spec['N'], seed)
    Lr = METHODS[spec['method']](spec, bench); N = bench.n_tasks
    Rm = np.full((N, N), np.nan); rows = []; t_run = time.time()
    for t in range(N):
        if DEV.type == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        Lr.pre_task(t); Lr.train_task(t); Lr.post_task(t)
        if DEV.type == 'cuda': torch.cuda.synchronize()
        dt = time.time() - t0
        if Lr.track_old: accs = [Lr.evaluate(s)[0] for s in range(t + 1)]
        else: accs = [np.nan] * t + [Lr.evaluate(t)[0]]
        Rm[t, :t + 1] = accs
        nxt = Lr.evaluate(t + 1)[0] if (t + 1 < N and Lr.track_old) else float('nan')
        bw = float(np.mean([Rm[t, s] - Rm[s, s] for s in range(t)])) if (t > 0 and Lr.track_old) else float('nan')
        row = dict(run_id=rid, task=t, acc_new=accs[t], acc_avg=float(np.nanmean(accs)), bwt_sofar=bw, acc_next_before=nxt,
                   time_task=dt, acc_row=accs)
        row.update(Lr.diag(t)); row.update(Lr.drift_stats(t)); rows.append(row)
    total = time.time() - t_run
    ACC = float(np.nanmean(Rm[N-1])) if Lr.track_old else float('nan')
    BWT = float(np.mean([Rm[N-1, s] - Rm[s, s] for s in range(N-1)])) if Lr.track_old else float('nan')
    meta = dict(run_id=rid, cfg_key=cfg_key(spec), bench=spec['bench'], N=N, width=spec['width'], method=spec['method'], hp=spec['hp'],
                seed=seed, label=spec['label'], tags=sorted(spec['tags']), exps=sorted(spec['exps']), ACC=ACC, BWT=BWT,
                acc_new_mean=float(np.nanmean(np.diag(Rm))), acc_new_mean_after0=float(np.nanmean(np.diag(Rm)[1:])) if N > 1 else float('nan'),
                time_total=total, extra_mem_floats=rows[-1].get('extra_mem_floats', 0), status='ok')
    evs = [dict(run_id=rid, **e) for e in getattr(Lr, 'events', [])]
    del Lr; gc.collect()
    if DEV.type == 'cuda': torch.cuda.empty_cache()
    return rows, meta, evs

def run_all(specs, max_hours=MAX_HOURS):
    done = {m['run_id'] for m in read_jsonl(F_META)}
    todo = [s for s in specs if run_id(s) not in done]
    print(f"{len(specs)} runs requested | {len(done)} already finished | {len(todo)} to do")
    t_start = time.time(); times = []
    for i, s in enumerate(todo):
        if max_hours and (time.time() - t_start) / 3600 > max_hours:
            print(f"Time budget of {max_hours} h reached - stopping. Re-run this cell to resume."); break
        t0 = time.time()
        try:
            rows, meta, evs = run_one(s)
            append_jsonl(F_TASK, rows); append_jsonl(F_EVT, evs); append_jsonl(F_META, [meta])
        except Exception as e:
            append_jsonl(F_ERR, [dict(run_id=run_id(s), cfg=cfg_key(s), seed=s['seed'], err=repr(e), tb=traceback.format_exc()[-1500:])])
            print("  ! run failed:", cfg_key(s)[:90], repr(e)[:120])
            gc.collect()
            if DEV.type == 'cuda': torch.cuda.empty_cache()
        times.append(time.time() - t0)
        if (i + 1) % 5 == 0 or i == len(todo) - 1:
            avg = np.mean(times[-30:]); eta = avg * (len(todo) - i - 1) / 3600
            print(f"[{i+1}/{len(todo)}] last={times[-1]:.0f}s avg={avg:.0f}s ETA~{eta:.1f}h | {s['label']} {s['bench']} d={s['width']} seed={s['seed']}")
    print("finished this pass.")


In [ ]:
#@title 8 · Experiment definitions (E1–E7)
SPECS = {}
EXP_ORDER = ['E1', 'E7', 'E6', 'E5', 'E3', 'E4', 'E2']

def add(exp, label, bench, N, width, method, hp=None, seeds=None):
    for sd in (seeds if seeds is not None else P['seeds']):
        s = dict(bench=bench, N=N, width=width, method=method, hp=dict(hp or {}), seed=sd, label=label, tags={label}, exps={exp})
        k = run_id(s)
        if k in SPECS: SPECS[k]['tags'].add(label); SPECS[k]['exps'].add(exp)
        else: SPECS[k] = s

SOCR_VARIANTS = {
    'SOCR-ref':     {},
    'SOCR-dense':   dict(gate='none', lam0=0.0, lam_r=0.0),
    'SOCR-B':       dict(reserve='input+write'),
    'SOCR-R2':      dict(recycle='R2'),
    'SOCR-R2R3':    dict(recycle='R2R3'),
    'SOCR-overlap': dict(recycle='overlap'),
}
BASELINES = {
    'FT': ('FT', {}), 'STL': ('STL', {}), 'ER': ('ER', dict(n_buf=200)), 'Parseval': ('Parseval', dict(lam=0.01)),
    'OGD': ('OGD', dict(n_g=20)), 'PackNet': ('PackNet', {}),
    'GPM': ('GPM', dict(variant='gpm')), 'SGP': ('GPM', dict(variant='sgp')), 'GPM-rand': ('GPM', dict(variant='rand')),
    'OLoRA-style': ('SOCR', dict(ortho='olora', gate='none', lam0=0.0, lam_r=0.0)),
}
def add_method(exp, label, bench, N, width):
    if label in BASELINES: m, hp = BASELINES[label]; add(exp, label, bench, N, width, m, hp)
    else: add(exp, label, bench, N, width, 'SOCR', SOCR_VARIANTS[label])

PM, PML = P['pm_tasks'], P['pm_tasks_long']
E = EXPERIMENTS

if E['E1']:   # ---------------- main comparison
    for lab in list(BASELINES) + list(SOCR_VARIANTS): add_method('E1', lab, 'pmnist', PM, 256)
    for nt in P['cifar_tasks']:
        for lab in list(BASELINES) + [k for k in SOCR_VARIANTS if k != 'SOCR-overlap']: add_method('E1', lab, 'cifar', nt, 256)

if E['E2']:   # ---------------- width sweep, N ~ 5 N_max
    for d in P['widths']:
        for lab in ['STL', 'GPM', 'SGP', 'PackNet'] + list(SOCR_VARIANTS): add_method('E2', lab, 'pmnist', PML, d)

if E['E3']:   # ---------------- one-factor-at-a-time ablation (d=128)
    ab = {'ABL:ref': {}, 'ABL:ortho=off': dict(ortho='off'), 'ABL:ortho=soft': dict(ortho='soft'),
          'ABL:sparsity=off': dict(gate='none', lam0=0.0, lam_r=0.0), 'ABL:sparsity=L0only': dict(lam_r=0.0),
          'ABL:sparsity=footprintonly': dict(gate='none', lam0=0.0),
          'ABL:recycle=R2': dict(recycle='R2'), 'ABL:recycle=R2R3': dict(recycle='R2R3'), 'ABL:recycle=overlap': dict(recycle='overlap'),
          'ABL:reserve=input+write': dict(reserve='input+write'),
          'ABL:frame=fixed_data': dict(frame='fixed_data'), 'ABL:frame=fixed_random': dict(frame='fixed_random'),
          'ABL:share_first': dict(share_first=True), 'ABL:base=random': dict(base_init='random')}
    for lab, hp in ab.items(): add('E3', lab, 'pmnist', PML, 128, 'SOCR', hp)
    for lab in ['GPM', 'SGP']: add_method('E3', lab, 'pmnist', PML, 128)
    if FACTORIAL:
        for o, sp, rc, rs, fr in itertools.product(['off', 'soft', 'hard'], ['off', 'L0', 'foot'], ['none', 'R2', 'R2R3', 'overlap'], ['input', 'input+write'], ['learned', 'fixed_random']):
            hp = dict(ortho=o, recycle=rc, reserve=rs, frame=fr)
            if sp == 'off': hp.update(gate='none', lam0=0.0, lam_r=0.0)
            elif sp == 'L0': hp.update(lam_r=0.0)
            else: hp.update(gate='none', lam0=0.0)
            add('E3', f"FACT:{o}|{sp}|{rc}|{rs}|{fr}", 'pmnist', PML, 128, 'SOCR', hp)

if E['E4']:   # ---------------- negative controls (d=128)
    ctrl = {'CTRL:randmask-nonortho': ('SOCR', dict(ortho='off', frame='fixed_gauss', gate='random_mask', rand_keep=0.5, lam0=0.0, lam_r=0.0)),
            'CTRL:random-projector': ('GPM', dict(variant='rand')),
            'CTRL:fixed-random-rotation': ('SOCR', dict(frame='fixed_random')),
            'CTRL:shuffled-assignment': ('SOCR', dict(shuffle_assign=True)),
            'CTRL:ortho-off': ('SOCR', dict(ortho='off'))}
    for lab, (m, hp) in ctrl.items(): add('E4', lab, 'pmnist', PML, 128, m, hp)
    add('E4', 'ABL:ref', 'pmnist', PML, 128, 'SOCR', {}); add('E4', 'GPM', 'pmnist', PML, 128, 'GPM', dict(variant='gpm'))

if E['E5']:   # ---------------- N > N_max recycling (H3)
    for d in [64, 128]:
        for lab in ['GPM', 'SGP'] + list(SOCR_VARIANTS): add_method('E5', lab, 'pmnist', PML, d)
        for ft in [0.5, 2.0]: add('E5', f'R2:free_target={ft}', 'pmnist', PML, d, 'SOCR', dict(recycle='R2', free_target=ft))
        for rr in [0.005, 0.1]: add('E5', f'R2R3:r2_rel={rr}', 'pmnist', PML, d, 'SOCR', dict(recycle='R2R3', r2_rel=rr))
        add('E5', 'R2R3:no-repair', 'pmnist', PML, d, 'SOCR', dict(recycle='R2R3', repair_steps=0))

if E['E6']:   # ---------------- footprint / L0 / eps sweeps (H2, Thm 2)  (d=128)
    for lr_ in [0.0, 0.02, 0.1, 0.5]:
        add('E6', f'FOOT:lam_r={lr_}', 'pmnist', PML, 128, 'SOCR', dict(gate='none', lam0=0.0, lam_r=lr_))
        add('E6', f'FOOT+sharefirst:lam_r={lr_}', 'pmnist', PML, 128, 'SOCR', dict(gate='none', lam0=0.0, lam_r=lr_, share_first=True))
    for l0 in [0.02, 0.5, 2.0]: add('E6', f'L0:lam0={l0}', 'pmnist', PML, 128, 'SOCR', dict(lam_r=0.0, lam0=l0))
    for ep in [0.90, 0.95, 0.99]: add('E6', f'EPS:eps={ep}', 'pmnist', PML, 128, 'SOCR', dict(eps=ep, gate='none', lam0=0.0, lam_r=0.0))
    for ep in [0.90, 0.95, 0.99]: add('E6', f'GPM-eps={ep}', 'pmnist', PML, 128, 'GPM', dict(variant='gpm', eps=ep))

if E['E7']:   # ---------------- learned vs fixed frame at several sparsities (H4)
    for bench, N, d in [('pmnist', PM, 128), ('cifar', P['cifar_tasks'][0], 256)]:
        for l0 in [0.02, 0.1, 0.5, 2.0]:
            for fr in ['learned', 'fixed_data', 'fixed_random']:
                add('E7', f'FRAME:{fr}|lam0={l0}', bench, N, d, 'SOCR', dict(frame=fr, lam0=l0, lam_r=0.0))

def spec_sort_key(s):
    ex = min(EXP_ORDER.index(e) for e in s['exps'])
    return (s['seed'], ex, s['bench'], s['width'], s['label'])
SPEC_LIST = sorted(SPECS.values(), key=spec_sort_key)
json.dump(dict(preset=PRESET, n_runs=len(SPEC_LIST), by_exp={e: sum(1 for s in SPEC_LIST if e in s['exps']) for e in EXP_ORDER}), open(os.path.join(OUT, "plan.json"), "w"), indent=1)
print("unique runs:", len(SPEC_LIST), "|", {e: sum(1 for s in SPEC_LIST if e in s['exps']) for e in EXP_ORDER})


In [ ]:
#@title 9 · RUN EXPERIMENTS E1–E7  (resumable — re-run this cell after a disconnect)
run_all(SPEC_LIST)


In [ ]:
#@title 10 · E8 — numerical checks of Theorems 1–3, gauge freedom, Welch bound, low-precision leakage
def theory_checks():
    out = {}; dt = torch.float64; torch.manual_seed(0)
    def fwd(Ws, x):
        h = x
        for i, W in enumerate(Ws):
            z = h @ W.t(); h = z if i == len(Ws) - 1 else torch.relu(z)
        return h
    def layer_inputs(Ws, x):
        ins = []; h = x
        for i, W in enumerate(Ws):
            ins.append(h); z = h @ W.t(); h = z if i == len(Ws) - 1 else torch.relu(z)
        return ins
    def basis(X, eps):
        U_, S_, Vh = torch.linalg.svd(X, full_matrices=False)
        e = torch.cumsum(S_ ** 2, 0) / (S_ ** 2).sum(); k = int((e < eps).sum().item()) + 1
        k = min(k, int((S_ > 1e-10 * S_[0]).sum().item())); return Vh[:k].t()
    dims = [64, 48, 32, 10]
    Ws = [torch.randn(dims[i+1], dims[i], dtype=dt) / math.sqrt(dims[i]) for i in range(3)]
    B0 = torch.linalg.qr(torch.randn(64, 20, dtype=dt))[0]
    xtr = torch.randn(300, 20, dtype=dt) @ B0.t(); xte = torch.randn(300, 20, dtype=dt) @ B0.t()
    # ---- Theorem 1 (exact isolation, eps=1) and control without projection
    ins = layer_inputs(Ws, xtr); Us = [basis(h, 1.0) for h in ins]
    rnd = [torch.randn_like(W) * 0.5 for W in Ws]
    dWp = [g - (g @ U) @ U.t() for g, U in zip(rnd, Us)]
    f0 = fwd(Ws, xtr)
    out['thm1'] = dict(max_abs_change_projected=float((fwd([W + d for W, d in zip(Ws, dWp)], xtr) - f0).abs().max()),
                       max_abs_change_unprojected=float((fwd([W + d for W, d in zip(Ws, rnd)], xtr) - f0).abs().max()),
                       ranks=[int(U.shape[1]) for U in Us])
    # ---- Theorem 2 (bound with truncated energy eps<1, held-out old-distribution inputs)
    t2 = []
    for eps in [1.0, 0.999, 0.99, 0.95, 0.9, 0.8]:
        Ue = [basis(h, eps) for h in ins]
        dW = [g - (g @ U) @ U.t() for g, U in zip(rnd, Ue)]; Wn = [W + d for W, d in zip(Ws, dW)]
        ins_te = layer_inputs(Ws, xte)
        epsl = [float((h - (h @ U) @ U.t()).norm(dim=1).max()) for h, U in zip(ins_te, Ue)]
        opn = [float(torch.linalg.matrix_norm(W, 2)) for W in Wn]; dn = [float(torch.linalg.matrix_norm(d, 2)) for d in dW]
        bound = sum(float(np.prod(opn[l+1:])) * dn[l] * epsl[l] for l in range(3))
        meas = float((fwd(Wn, xte) - fwd(Ws, xte)).norm(dim=1).max())
        t2.append(dict(eps=eps, measured=meas, bound=bound, ratio=meas / bound if bound > 0 else None, holds=bool(meas <= bound * (1 + 1e-9) + 1e-12)))
    out['thm2'] = t2
    # ---- Theorem 3 (soft projector)
    d, m, r = 40, 12, 10
    U = torch.linalg.qr(torch.randn(d, r, dtype=dt))[0]; lam = torch.rand(r, dtype=dt); Gm = torch.randn(m, d, dtype=dt); eta = 0.3
    dW = -eta * (Gm - (Gm @ U) * lam[None] @ U.t()); resid = (dW @ U + eta * (1 - lam)[None] * (Gm @ U)).abs().max().item()
    xs = U @ torch.randn(r, dtype=dt) + (torch.eye(d, dtype=dt) - U @ U.t()) @ torch.randn(d, dtype=dt)
    c_ = U.t() @ xs; rr = xs - U @ c_
    out['thm3'] = dict(max_resid_identity=resid, lhs=float((dW @ xs).norm()), rhs=float(eta * torch.linalg.matrix_norm(Gm, 2) * ((1 - lam) * c_).norm() + eta * torch.linalg.matrix_norm(Gm, 2) * rr.norm()))
    # ---- Gauge freedom and L0 non-invariance
    m, d, p = 30, 50, 12; Sm = torch.randn(m, p, dtype=dt) * (torch.rand(m, p, dtype=dt) < 0.2); Qm = torch.linalg.qr(torch.randn(d, p, dtype=dt))[0]
    Rg = torch.linalg.qr(torch.randn(p, p, dtype=dt))[0]
    out['gauge'] = dict(product_diff=float(((Sm @ Rg) @ (Qm @ Rg).t() - Sm @ Qm.t()).abs().max()),
                        nnz_frac_S=float((Sm != 0).double().mean()), nnz_frac_S_rotated=float(((Sm @ Rg).abs() > 1e-12).double().mean()))
    # ---- weight-space orthogonality neither necessary nor sufficient
    d = 40; Us_ = torch.linalg.qr(torch.randn(d, 8, dtype=dt))[0]; k = 4
    St, Ss = torch.randn(10, k, dtype=dt), torch.randn(10, k, dtype=dt)
    Qt = torch.linalg.qr((torch.eye(d, dtype=dt) - Us_ @ Us_.t()) @ torch.randn(d, k, dtype=dt))[0]; Qs = torch.linalg.qr(torch.randn(d, k, dtype=dt))[0]
    dWt, dWs = St @ Qt.t(), Ss @ Qs.t()
    case1 = dict(inner_product=float((dWt * dWs).sum()), interference_on_old=float((dWt @ Us_).norm()))
    Qt2 = Us_[:, :k]; Qs2 = torch.linalg.qr((torch.eye(d, dtype=dt) - Qt2 @ Qt2.t()) @ torch.randn(d, k, dtype=dt))[0]
    case2 = dict(inner_product=float(((St @ Qt2.t()) * (Ss @ Qs2.t())).sum()), interference_on_old=float(((St @ Qt2.t()) @ Us_).norm()))
    out['orth_not_nec_not_suff'] = dict(isolated_but_not_weight_orthogonal=case1, weight_orthogonal_but_not_isolated=case2)
    # ---- Welch bound / coherence of N unit vectors in R^d
    wel = []
    for d in [16, 32, 64]:
        for mult in [1, 2, 4, 8]:
            Nn = d * mult
            if Nn > 512: continue
            mus = []
            for _ in range(3):
                V = torch.randn(d, Nn, dtype=dt); V = V / V.norm(dim=0); G_ = (V.t() @ V).abs() - torch.eye(Nn, dtype=dt); mus.append(float(G_.max()))
            V = torch.randn(d, Nn, dtype=torch.float32, requires_grad=True); opt = torch.optim.Adam([V], lr=0.05)
            mask = ~torch.eye(Nn, dtype=torch.bool)
            for _ in range(250):
                Vn = V / V.norm(dim=0); G_ = (Vn.t() @ Vn).abs()[mask]; loss = torch.logsumexp(60 * G_, 0) / 60
                opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                Vn = V / V.norm(dim=0); mu_opt = float((Vn.t() @ Vn).abs()[mask].max())
            welch = math.sqrt(max(Nn - d, 0) / (d * (Nn - 1))) if Nn > d else 0.0
            wel.append(dict(d=d, N=Nn, mu_random=float(np.mean(mus)), mu_optimised=mu_opt, welch=welch, inv_sqrt_d=1 / math.sqrt(d)))
    out['welch'] = wel
    # ---- low-precision projection leakage
    def leak_sim(wdtype, pdtype, restore_every, steps=400, d=256, m=64, r=64, lr=1e-2):
        torch.manual_seed(1)
        Uq = torch.linalg.qr(torch.randn(d, r))[0]; W = (torch.randn(m, d) * 0.1).to(wdtype); Wref = W.clone(); tr = []
        for k in range(steps):
            g = torch.randn(m, d); gp = g.to(pdtype); Up = Uq.to(pdtype); gp = gp - (gp @ Up) @ Up.t()
            W = (W.float() - lr * gp.float()).to(wdtype)
            if restore_every and (k + 1) % restore_every == 0:
                D = W.float() - Wref.float(); D = D - (D @ Uq) @ Uq.t(); W = (Wref.float() + D).to(wdtype)
            if (k + 1) % 50 == 0: tr.append(float(((W.float() - Wref.float()) @ Uq).norm()))
        return tr
    leak = {}
    for name, (wd, pd_, re) in {'fp32/fp32': (torch.float32, torch.float32, 0), 'bf16 weights, bf16 projection': (torch.bfloat16, torch.bfloat16, 0),
                                'bf16 weights, fp32 projection': (torch.bfloat16, torch.float32, 0), 'bf16 weights, fp32 projection, restore/100': (torch.bfloat16, torch.float32, 100),
                                'fp16 weights, fp16 projection': (torch.float16, torch.float16, 0)}.items():
        try: leak[name] = leak_sim(wd, pd_, re)
        except Exception as e: leak[name] = f"error: {e!r}"
    out['leakage_every50_steps'] = leak
    # ---- orthonormality drift of an incrementally grown basis
    drift = {}
    d, ncol = 256, 200; torch.manual_seed(2); base = torch.randn(d, 8); cols = [base @ torch.randn(8) + 1e-3 * torch.randn(d) for _ in range(ncol)]
    def grow(kind):
        U = torch.zeros(d, 0); errs = []
        for i, v in enumerate(cols):
            v = v.clone()
            if kind == 'cgs1_fp32': v = v - U @ (U.t() @ v)
            elif kind == 'cgs2_fp32': v = v - U @ (U.t() @ v); v = v - U @ (U.t() @ v)
            elif kind == 'cgs2_fp64+reortho':
                v = v.double(); U = U.double(); v = v - U @ (U.t() @ v); v = v - U @ (U.t() @ v)
            nrm = v.norm()
            if nrm < 1e-6: continue
            U = torch.cat([U, (v / nrm).to(U.dtype).unsqueeze(1)], 1)
            if kind == 'cgs2_fp64+reortho' and (i + 1) % 20 == 0: U = torch.linalg.qr(U)[0]
            if (i + 1) % 50 == 0: errs.append(float((U.t().double() @ U.double() - torch.eye(U.shape[1], dtype=torch.float64)).norm()))
        return dict(final_cols=int(U.shape[1]), err_every50=errs)
    for kd in ['cgs1_fp32', 'cgs2_fp32', 'cgs2_fp64+reortho']: drift[kd] = grow(kd)
    out['basis_drift'] = drift
    return out

if EXPERIMENTS['E8']:
    TH = theory_checks()
    json.dump(clean(TH), open(os.path.join(OUT, "theory_checks.json"), "w"), indent=1)
    print("Thm1 projected vs unprojected max |Δf|:", TH['thm1'])
    print("Thm2 ratios (measured/bound):", [(r['eps'], round(r['ratio'], 4) if r['ratio'] is not None else None, r['holds']) for r in TH['thm2']])
    print("Thm3:", TH['thm3']); print("Gauge:", TH['gauge'])
    print("Orth neither necessary nor sufficient:", TH['orth_not_nec_not_suff'])
    print("Welch:", [(w['d'], w['N'], round(w['mu_random'], 3), round(w['mu_optimised'], 3), round(w['welch'], 3)) for w in TH['welch']])
    print("saved theory_checks.json")


In [ ]:
#@title 11 · Load results + main comparison table (E1)
import re, scipy.stats as st
import matplotlib; import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)

runs = pd.DataFrame(read_jsonl(F_META)); rows = pd.DataFrame(read_jsonl(F_TASK)); evs = pd.DataFrame(read_jsonl(F_EVT)); errs = read_jsonl(F_ERR)
print(f"{len(runs)} finished runs | {len(rows)} task rows | {len(evs)} recycling events | {len(errs)} failed runs")
if errs: print("first failure:", errs[0]['cfg'][:100], errs[0]['err'][:200])
SUMMARY = dict(preset=PRESET, n_runs=len(runs), n_failed=len(errs))
if len(runs):
    runs['tagset'] = runs['tags'].apply(set); runs['expset'] = runs['exps'].apply(set)
    R = rows.merge(runs[['run_id', 'label', 'bench', 'N', 'width', 'method', 'seed', 'cfg_key']], on='run_id')

def sel(tag=None, exp=None, bench=None, width=None, N=None, df=None):
    d = runs if df is None else df
    if tag is not None:  d = d[d.tagset.apply(lambda s: tag in s)]
    if exp is not None:  d = d[d.expset.apply(lambda s: exp in s)]
    if bench is not None: d = d[d.bench == bench]
    if width is not None: d = d[d.width == width]
    if N is not None: d = d[d.N == N]
    return d

def paired(a, b, col, scale=1.0):
    m = a[['seed', col]].merge(b[['seed', col]], on='seed', suffixes=('_a', '_b')).dropna()
    if len(m) < 2: return dict(n=len(m), diff=float('nan'), ci_lo=float('nan'), ci_hi=float('nan'), p_t=float('nan'), p_w=float('nan'))
    d = (m[col + '_a'] - m[col + '_b']).values * scale; n = len(d); se = d.std(ddof=1) / math.sqrt(n)
    tcrit = st.t.ppf(0.975, n - 1)
    try: pw = st.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    except Exception: pw = float('nan')
    return dict(n=n, diff=float(d.mean()), ci_lo=float(d.mean() - tcrit * se), ci_hi=float(d.mean() + tcrit * se),
                p_t=float(st.ttest_1samp(d, 0).pvalue) if d.std() > 0 else float('nan'), p_w=float(pw))

free_cols = [c for c in (R.columns if len(runs) else []) if re.match(r'^free_l\d+$', c)]
def exhaustion(g, hidden_only=False, thr=4):
    """number of tasks fully allocated before any (hidden) layer's free dimension drops below thr; N+1 = never exhausted (censored)."""
    g = g.sort_values('task'); cols = [c for c in free_cols if g[c].notna().any() and not (hidden_only and c == 'free_l0')]
    if not cols: return float('nan')
    m = g[cols].min(axis=1).values; idx = np.where(m < thr)[0]
    return float(idx[0] + 1) if len(idx) else float(len(g) + 1)
def add_run_features():
    ex_all = R.groupby('run_id').apply(lambda g: exhaustion(g)).rename('n_max_obs')
    ex_hid = R.groupby('run_id').apply(lambda g: exhaustion(g, True)).rename('n_max_hidden')
    spars = R[R.task >= 1].groupby('run_id')['sparsity_mean'].mean().rename('sparsity') if 'sparsity_mean' in R else None
    hid_cols = [c for c in R.columns if re.match(r'^foot_l[1-9]$', c)]
    early = R[(R.task >= 1) & (R.task <= 5)]
    rbar = early.groupby('run_id')[hid_cols].mean().mean(axis=1).rename('rbar_hidden') if hid_cols else None
    r0 = early.groupby('run_id')['foot_l0'].mean().rename('rbar_l0') if 'foot_l0' in early else None
    acc5 = early.groupby('run_id')['acc_new'].mean().rename('acc_new_t1_5')
    out = runs.set_index('run_id')
    for s in [ex_all, ex_hid, spars, rbar, r0, acc5]:
        if s is not None: out = out.join(s)
    return out.reset_index()
if len(runs):
    runs = add_run_features(); runs['tagset'] = runs['tags'].apply(set); runs['expset'] = runs['exps'].apply(set)
    # plasticity relative to single-task learning from scratch
    stl = runs[runs.method == 'STL'].groupby(['bench', 'N', 'width', 'seed'])['acc_new_mean_after0'].mean().rename('stl_acc').reset_index()
    runs = runs.merge(stl, on=['bench', 'N', 'width', 'seed'], how='left'); runs['plast_ratio'] = runs['acc_new_mean_after0'] / runs['stl_acc']

    e1 = sel(exp='E1')
    tab = e1.groupby(['bench', 'N', 'width', 'label']).agg(ACC=('ACC', 'mean'), ACC_sd=('ACC', 'std'), BWT=('BWT', 'mean'), BWT_sd=('BWT', 'std'),
            plast_ratio=('plast_ratio', 'mean'), time_s=('time_total', 'mean'), extra_floats=('extra_mem_floats', 'mean'), n_seeds=('seed', 'count')).reset_index()
    for c in ['ACC', 'ACC_sd', 'BWT', 'BWT_sd']: tab[c] = (tab[c] * 100).round(2)
    tab['plast_ratio'] = tab['plast_ratio'].round(3); tab['time_s'] = tab['time_s'].round(0)
    print("\n=== E1 main comparison (percent; plast_ratio = acc on new tasks / single-task-from-scratch acc) ===")
    print(tab.to_string(index=False)); tab.to_csv(os.path.join(OUT, "table_E1_main.csv"), index=False)
    SUMMARY['E1_table'] = tab.to_dict(orient='records')


In [ ]:
#@title 12 · Hypothesis tests H1–H4, controls, ablations, capacity law
def safe(name, fn):
    try: fn()
    except Exception as e:
        print(f"[{name}] analysis failed: {e!r}"); SUMMARY[name + '_error'] = repr(e); traceback.print_exc(limit=2)
def pct(x): return None if x is None or (isinstance(x, float) and math.isnan(x)) else round(100 * x, 2)

# ------------------------------------------------------------------ H1
def H1():
    print("\n================ H1: BWT ≈ 0 and drift within the Theorem-2 bound for N ≤ N_max ================")
    out = []
    for lab in ['SOCR-ref', 'SOCR-dense', 'SOCR-B', 'GPM']:
        for bench in ['pmnist', 'cifar']:
            rs = sel(tag=lab, bench=bench, exp='E1')
            for _, r in rs.iterrows():
                g = R[(R.run_id == r.run_id)].sort_values('task'); nmax = r.n_max_obs
                if not (nmax == nmax): continue
                last = int(min(nmax, r.N)) - 1                      # last task learned with free capacity
                if last < 1: continue
                bwt = g.loc[g.task == last, 'bwt_sofar'].values
                w = g[(g.task >= 1) & (g.task <= last)]
                if 'thm2_bound' in w and 'drift_max' in w:
                    if 'released_now' in w: w = w[w['released_now'].fillna(0) == 0]
                    w = w[w.thm2_bound.notna() & (w.thm2_bound > 0)]
                    ratio = (w.drift_max / w.thm2_bound).values
                else: ratio = np.array([])
                out.append(dict(label=lab, bench=bench, seed=r.seed, last_task=last, censored=bool(nmax > r.N), bwt_at_last=float(bwt[0]) if len(bwt) else np.nan,
                                max_ratio=float(ratio.max()) if len(ratio) else np.nan, frac_within=float((ratio <= 1 + 1e-6).mean()) if len(ratio) else np.nan,
                                median_ratio=float(np.median(ratio)) if len(ratio) else np.nan,
                                leak=float(np.nanmax([g[c].max() for c in g.columns if re.match(r'^leak_l\d+$', c)])) if any(re.match(r'^leak_l\d+$', c) for c in g.columns) else np.nan))
    if not out: print("no runs available"); return
    d = pd.DataFrame(out)
    t = d.groupby(['label', 'bench']).agg(BWT_pp=('bwt_at_last', lambda x: 100 * x.mean()), BWT_sd=('bwt_at_last', lambda x: 100 * x.std()),
        frac_within_bound=('frac_within', 'mean'), median_drift_over_bound=('median_ratio', 'median'), max_drift_over_bound=('max_ratio', 'max'),
        max_leak=('leak', 'max'), n_seeds=('seed', 'count'), mean_last_task=('last_task', 'mean')).round(4).reset_index()
    print(t.to_string(index=False)); t.to_csv(os.path.join(OUT, "table_H1.csv"), index=False); SUMMARY['H1_table'] = t.to_dict(orient='records')
    ok = []
    for lab in ['SOCR-ref', 'SOCR-dense', 'SOCR-B']:
        x = t[t.label == lab]
        if len(x): ok.append(bool((x.BWT_pp.abs() <= 1.0).all() and (x.frac_within_bound >= 0.99).all()))
    SUMMARY['H1_verdict'] = 'SUPPORTED' if ok and all(ok) else ('PARTIAL/NOT SUPPORTED' if ok else 'NO DATA')
    print("H1 criterion: |BWT| <= 1pp and >=99% of updates within bound for every SOCR variant ->", SUMMARY['H1_verdict'])
safe('H1', H1)

# ------------------------------------------------------------------ H2
def H2():
    print("\n================ H2: footprint regularisation lowers r̄ and raises N_max at equal accuracy ================")
    base = sel(tag='FOOT:lam_r=0.0', bench='pmnist'); out = []
    for variant in ['FOOT', 'FOOT+sharefirst']:
        b0 = sel(tag=f'{variant}:lam_r=0.0', bench='pmnist')
        for lr_ in [0.02, 0.1, 0.5]:
            a = sel(tag=f'{variant}:lam_r={lr_}', bench='pmnist')
            if not len(a) or not len(b0): continue
            o = dict(variant=variant, lam_r=lr_)
            for col, scale in [('rbar_hidden', 1), ('rbar_l0', 1), ('n_max_hidden', 1), ('n_max_obs', 1), ('acc_new_t1_5', 100), ('ACC', 100), ('BWT', 100)]:
                p = paired(a, b0, col, scale); o[col + '_diff'] = round(p['diff'], 3); o[col + '_p'] = round(p['p_t'], 4) if p['p_t'] == p['p_t'] else None
                if col == 'acc_new_t1_5': o['acc_ci_lo'] = round(p['ci_lo'], 3)
            o['rbar_hidden_base'] = round(float(b0.rbar_hidden.mean()), 2); o['n'] = len(a); out.append(o)
    if not out: print("no data"); return
    d = pd.DataFrame(out); print(d.to_string(index=False)); d.to_csv(os.path.join(OUT, "table_H2.csv"), index=False); SUMMARY['H2_table'] = d.to_dict(orient='records')
    good = d[(d.rbar_hidden_diff < 0) & (d.rbar_hidden_p.fillna(1) < 0.05) & (d.n_max_hidden_diff >= 0) & (d.acc_ci_lo > -1.0)]
    SUMMARY['H2_verdict'] = 'SUPPORTED (for hidden layers) at lam_r=' + str(sorted(good.lam_r.unique().tolist())) if len(good) else 'NOT SUPPORTED'
    ov = d[d.variant == 'FOOT']
    SUMMARY['H2_note'] = 'overall N_max (all layers) change vs lam_r=0 (mean paired diff per lam_r): ' + str(dict(zip(ov.lam_r, ov.n_max_obs_diff)))
    print("H2 criterion: r̄(hidden) lower (p<.05), N_max(hidden) not lower, accuracy non-inferior (CI lower bound > -1pp) ->", SUMMARY['H2_verdict']); print(SUMMARY['H2_note'])
safe('H2', H2)

# ------------------------------------------------------------------ H3
def H3():
    print("\n================ H3: loss per released dimension stays below (Σ√ι)² ================")
    if not len(evs): print("no recycling events logged"); return
    e = evs.merge(runs[['run_id', 'label', 'width', 'seed', 'bench']], on='run_id'); r2 = e[(e.stage == 'R2') & e.pred_bound.notna()].copy()
    if not len(r2): print("no R2 events"); return
    r2['holds'] = r2.meas_dloss_sum <= r2.pred_bound * 1.0 + 1e-9; r2['ratio'] = r2.meas_dloss_sum / r2.pred_bound.clip(lower=1e-12)
    rho = st.spearmanr(r2.pred_bound, r2.meas_dloss_sum).correlation if len(r2) > 3 else float('nan')
    print(f"R2 events: {len(r2)} | fraction with measured ΔL_old <= bound: {r2.holds.mean():.3f} | median measured/bound: {r2.ratio.median():.3f} | 90th pct: {r2.ratio.quantile(.9):.3f} | Spearman(pred, measured): {rho:.3f}")
    print(r2.groupby(['width', 'label']).agg(n=('holds', 'size'), frac_holds=('holds', 'mean'), med_ratio=('ratio', 'median'), mean_released=('n_released', 'mean'),
          mean_acc_drop_pp=('acc_drop_mean', lambda x: 100 * x.mean())).round(3).to_string())
    SUMMARY['H3'] = dict(n_events=int(len(r2)), frac_within_bound=float(r2.holds.mean()), median_ratio=float(r2.ratio.median()), spearman=float(rho))
    SUMMARY['H3_verdict'] = 'SUPPORTED' if (r2.holds.mean() >= 0.95 and rho > 0.3) else 'NOT SUPPORTED / INCONCLUSIVE'
    print("H3 criterion: >=95% of events within bound and Spearman > 0.3 ->", SUMMARY['H3_verdict'])
    r3 = e[e.stage == 'R3']
    if len(r3):
        print("\nR3 (energy-ranked release + pseudo-input repair): mean total ΔL_old before / after repair:",
              round(r3.total_dloss_prerepair.mean(), 4), '/', round(r3.total_dloss_postrepair.mean(), 4), "| acc drop pp before / after:",
              round(100 * r3.acc_drop_total_prerepair.mean(), 2), '/', round(100 * r3.acc_drop_total_postrepair.mean(), 2))
        SUMMARY['R3'] = dict(dloss_pre=float(r3.total_dloss_prerepair.mean()), dloss_post=float(r3.total_dloss_postrepair.mean()))
    r2.to_csv(os.path.join(OUT, "table_H3_events.csv"), index=False)
    # beyond N_max: does recycling restore plasticity?
    print("\n--- beyond N_max (E5): plasticity of the last third of tasks, final ACC, BWT ---")
    q = []
    for _, r in runs[runs.expset.apply(lambda s: 'E5' in s)].iterrows():
        g = R[R.run_id == r.run_id].sort_values('task'); k = max(1, len(g) // 3)
        q.append(dict(width=r.width, label=r.label, seed=r.seed, late_acc_new=g.acc_new.tail(k).mean(), ACC=r.ACC, BWT=r.BWT, n_max=r.n_max_obs))
    if q:
        qd = pd.DataFrame(q).groupby(['width', 'label']).mean(numeric_only=True).drop(columns='seed')
        for c in ['late_acc_new', 'ACC', 'BWT']: qd[c] = (100 * qd[c]).round(2)
        print(qd.to_string()); qd.reset_index().to_csv(os.path.join(OUT, "table_E5_beyond_nmax.csv"), index=False); SUMMARY['E5_table'] = qd.reset_index().to_dict(orient='records')
safe('H3', H3)

# ------------------------------------------------------------------ H4
def H4():
    print("\n================ H4: learned frame vs fixed frame at equal sparsity ================")
    f = runs[runs.tagset.apply(lambda s: any(t.startswith('FRAME:') for t in s))].copy()
    if not len(f): print("no data"); return
    f['frame'] = f.hp.apply(lambda h: h.get('frame')); f['lam0'] = f.hp.apply(lambda h: h.get('lam0'))
    t = f.groupby(['bench', 'lam0', 'frame']).agg(ACC=('ACC', 'mean'), BWT=('BWT', 'mean'), acc_new=('acc_new_mean_after0', 'mean'), sparsity=('sparsity', 'mean'), n=('seed', 'count')).reset_index()
    for c in ['ACC', 'BWT', 'acc_new']: t[c] = (100 * t[c]).round(2)
    t['sparsity'] = t.sparsity.round(3); print(t.to_string(index=False)); t.to_csv(os.path.join(OUT, "table_H4.csv"), index=False); SUMMARY['H4_table'] = t.to_dict(orient='records')
    res = []
    for bench in f.bench.unique():
        for l0 in sorted(f.lam0.unique()):
            L_ = f[(f.bench == bench) & (f.lam0 == l0) & (f.frame == 'learned')]
            for other in ['fixed_data', 'fixed_random']:
                O = f[(f.bench == bench) & (f.lam0 == l0) & (f.frame == other)]
                p = paired(L_, O, 'ACC', 100); ps = paired(L_, O, 'sparsity', 1)
                res.append(dict(bench=bench, lam0=l0, vs=other, dACC_pp=round(p['diff'], 3), ci_lo=round(p['ci_lo'], 3), ci_hi=round(p['ci_hi'], 3), p_t=round(p['p_t'], 4) if p['p_t'] == p['p_t'] else None, d_sparsity=round(ps['diff'], 3), n=p['n']))
    rd = pd.DataFrame(res); print("\npaired learned − fixed (ACC, percentage points):"); print(rd.to_string(index=False)); SUMMARY['H4_paired'] = rd.to_dict(orient='records')
    # equal-sparsity regression ACC ~ 1 + sparsity + learned, bootstrap CI
    verdicts = {}
    rng = np.random.default_rng(0)
    for bench in f.bench.unique():
        for other in ['fixed_data', 'fixed_random']:
            d = f[(f.bench == bench) & (f.frame.isin(['learned', other]))].dropna(subset=['ACC', 'sparsity'])
            if len(d) < 8: continue
            def coef(dd):
                X = np.c_[np.ones(len(dd)), dd.sparsity.values, (dd.frame == 'learned').astype(float).values]
                return np.linalg.lstsq(X, dd.ACC.values * 100, rcond=None)[0][2]
            c0 = coef(d); bs = [coef(d.iloc[rng.integers(0, len(d), len(d))]) for _ in range(1000)]
            lo, hi = np.percentile(bs, [2.5, 97.5]); verdicts[f'{bench}:learned_vs_{other}'] = dict(effect_pp=round(float(c0), 3), ci=[round(float(lo), 3), round(float(hi), 3)], supports_H4=bool(lo > 0))
    print("\nequal-sparsity adjusted effect of 'learned' (pp ACC):"); [print(' ', k, v) for k, v in verdicts.items()]
    SUMMARY['H4_adjusted'] = verdicts
    SUMMARY['H4_verdict'] = 'SUPPORTED' if verdicts and all(v['supports_H4'] for k, v in verdicts.items() if 'fixed_random' in k) else 'NOT SUPPORTED (Stiefel/learned frame not shown to help)'
    print("H4 verdict (learned beats fixed_random with CI>0 on every benchmark):", SUMMARY['H4_verdict'])
safe('H4', H4)

# ------------------------------------------------------------------ controls and ablations
def CTRL_ABL():
    print("\n================ E4 negative controls vs SOCR reference (d=128, long sequence) ================")
    ref = sel(tag='ABL:ref', bench='pmnist', width=128); out = []
    for lab in [t for t in sorted(set().union(*runs.tagset)) if t.startswith('CTRL:') or t.startswith('ABL:') and t != 'ABL:ref'] + ['GPM', 'SGP']:
        a = sel(tag=lab, bench='pmnist', width=128, N=P['pm_tasks_long'])
        if lab in ('GPM', 'SGP'): a = a[a.exps.apply(lambda e: 'E3' in e or 'E4' in e)]
        if not len(a) or not len(ref): continue
        o = dict(config=lab, n=len(a), ACC=pct(a.ACC.mean()), BWT=pct(a.BWT.mean()), plast=pct(a.acc_new_mean_after0.mean()), sparsity=round(float(a.sparsity.mean()), 3) if a.sparsity.notna().any() else None, n_max=round(float(a.n_max_obs.mean()), 1) if a.n_max_obs.notna().any() else None)
        for col in ['ACC', 'BWT']:
            p = paired(a, ref, col, 100); o[f'd{col}_vs_ref_pp'] = round(p['diff'], 2); o[f'p_{col}'] = round(p['p_t'], 4) if p['p_t'] == p['p_t'] else None
        out.append(o)
    d = pd.DataFrame(out)
    if len(d): print(d.to_string(index=False)); d.to_csv(os.path.join(OUT, "table_E3_E4_ablation_controls.csv"), index=False); SUMMARY['E3_E4_table'] = d.to_dict(orient='records')
safe('CTRL_ABL', CTRL_ABL)

# ------------------------------------------------------------------ capacity law (E2)
def CAP():
    print("\n================ E2: capacity law N_max ≈ d_l / r̄_l ================")
    out = []
    for _, r in runs[runs.expset.apply(lambda s: 'E2' in s) & runs.method.isin(['GPM', 'SOCR'])].iterrows():
        g = R[R.run_id == r.run_id].sort_values('task'); o = dict(width=r.width, label=r.label, seed=r.seed, n_max_obs=r.n_max_obs, censored=bool(r.n_max_obs > r.N))
        th = []
        for l in range(4):
            if f'foot_l{l}' in g and g[f'foot_l{l}'].notna().any() and not (l == 0 and r.hp.get('share_first')):
                rb = g[(g.task >= 1) & (g.task <= 3)][f'foot_l{l}'].mean(); D = [784, r.width, r.width, 10][l]
                th.append(D / rb if rb > 0 else np.inf)
        o['n_max_theory'] = float(min(th)) if th else np.nan; out.append(o)
    if not out: print("no data"); return
    d = pd.DataFrame(out).groupby(['width', 'label']).mean(numeric_only=True).drop(columns='seed').round(2); print(d.to_string())
    d.reset_index().to_csv(os.path.join(OUT, "table_E2_capacity.csv"), index=False); SUMMARY['E2_table'] = d.reset_index().to_dict(orient='records')
safe('CAP', CAP)

json.dump(clean(SUMMARY), open(os.path.join(OUT, "summary.json"), "w"), indent=1)
print("\nverdict summary:", {k: v for k, v in SUMMARY.items() if k.endswith('_verdict')})


In [ ]:
#@title 13 · Figures + export bundle (zip) for analysis
def savefig(name): plt.tight_layout(); plt.savefig(os.path.join(OUT, "figs", name), dpi=130); plt.show()

def fig_e1():
    d = sel(exp='E1', bench='pmnist'); d = d[d.width == 256]
    if not len(d): return
    g = d.groupby('label').agg(ACC=('ACC', 'mean'), ACC_sd=('ACC', 'std'), BWT=('BWT', 'mean'), BWT_sd=('BWT', 'std')).drop(index='STL', errors='ignore').sort_values('ACC')
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].barh(g.index, 100 * g.ACC, xerr=100 * g.ACC_sd.fillna(0)); ax[0].set_title("Permuted-MNIST final ACC (%)")
    ax[1].barh(g.index, 100 * g.BWT, xerr=100 * g.BWT_sd.fillna(0), color='tab:red'); ax[1].set_title("BWT (pp)"); savefig("fig_E1_pmnist.png")
def fig_plasticity():
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, w in zip(axs, [64, 128]):
        for lab in ['GPM', 'SGP', 'SOCR-ref', 'SOCR-R2', 'SOCR-R2R3', 'SOCR-overlap']:
            rs = sel(tag=lab, bench='pmnist', width=w, N=P['pm_tasks_long'])
            if not len(rs): continue
            g = R[R.run_id.isin(rs.run_id)].groupby('task')['acc_new'].mean(); ax.plot(g.index, 100 * g.values, label=lab)
        ax.set_title(f"accuracy on the task just learned, d={w} (capacity exhaustion → drop)"); ax.set_xlabel("task index"); ax.set_ylabel("%"); ax.legend(fontsize=8)
    savefig("fig_plasticity_vs_tasks.png")
def fig_free():
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.5))
    for lab in ['GPM', 'SOCR-ref', 'SOCR-R2']:
        rs = sel(tag=lab, bench='pmnist', width=128, N=P['pm_tasks_long'])
        if not len(rs) or 'free_l1' not in R: continue
        g = R[R.run_id.isin(rs.run_id)].groupby('task')['free_l1'].mean(); ax.plot(g.index, g.values, label=lab)
    ax.set_title("free dimensions in hidden layer 1 (d=128)"); ax.set_xlabel("task"); ax.legend(); savefig("fig_free_dims.png")
def fig_h3():
    if not len(evs): return
    e = evs[(evs.stage == 'R2') & evs.pred_bound.notna()]
    if not len(e): return
    plt.figure(figsize=(5.5, 5)); plt.loglog(e.pred_bound.clip(lower=1e-9), e.meas_dloss_sum.clip(lower=1e-9), '.', alpha=.5)
    lim = [1e-9, max(e.pred_bound.max(), e.meas_dloss_sum.max())]; plt.plot(lim, lim, 'k--', label='measured = bound'); plt.xlabel("predicted bound (Σ√ι)²"); plt.ylabel("measured ΔL_old (sum over old tasks)"); plt.legend(); plt.title("H3"); savefig("fig_H3_scatter.png")
def fig_h4():
    f = runs[runs.tagset.apply(lambda s: any(t.startswith('FRAME:') for t in s))].copy()
    if not len(f): return
    f['frame'] = f.hp.apply(lambda h: h.get('frame')); benches = f.bench.unique()
    fig, axs = plt.subplots(1, len(benches), figsize=(6 * len(benches), 4.5), squeeze=False)
    for ax, b in zip(axs[0], benches):
        for fr, grp in f[f.bench == b].groupby('frame'):
            ax.scatter(grp.sparsity, 100 * grp.ACC, label=fr, alpha=.7)
        ax.set_title(f"H4 {b}: ACC vs sparsity"); ax.set_xlabel("mean gate sparsity"); ax.set_ylabel("ACC %"); ax.legend()
    savefig("fig_H4_frame.png")
if len(runs):
    for fn in [fig_e1, fig_plasticity, fig_free, fig_h3, fig_h4]:
        try: fn()
        except Exception as e: print(fn.__name__, "failed:", repr(e))

    # ---------- flat CSV exports + zip
    flat = R.copy(); flat['acc_row'] = flat['acc_row'].apply(json.dumps)
    flat.to_csv(os.path.join(OUT, "task_rows_flat.csv"), index=False)
    rr = runs.drop(columns=['tagset', 'expset'], errors='ignore').copy()
    for c in ['hp', 'tags', 'exps']: rr[c] = rr[c].apply(json.dumps)
    rr.to_csv(os.path.join(OUT, "runs_summary.csv"), index=False)
    if len(evs): evs.to_csv(os.path.join(OUT, "events_flat.csv"), index=False)
    json.dump(clean(SUMMARY), open(os.path.join(OUT, "summary.json"), "w"), indent=1)
zp = f"/content/socr_results_{PRESET}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), OUT))
print("bundle:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(zp)
except Exception as e: print("download manually from the file browser:", zp)
print("\nUpload socr_results_%s.zip (or at least summary.json, runs_summary.csv, task_rows_flat.csv, events_flat.csv, theory_checks.json) back to the chat for analysis." % PRESET)
